# Умный пайплайн подготовки данных (Auto-Preprocessor)
Этот ноутбук демонстрирует работу моего кастомного модуля `core.py`. Скрипт автоматизирует рутину аналитика: сам находит скрытые категории, парсит даты, умно заполняет пропуски, удаляет выбросы через ML и борется с мультиколлинеарностью. 

В качестве примера используем классический «грязный» датасет House Prices.

In [ ]:
import pandas as pd

from core import (
    convert_hidden_categories, drop_columns_by_index, parse_dt, 
    drop_useless_columns_smartly, fill_nan_with_new_class, 
    fill_nan_smart_impute, two_dimens, zero_corr, 
    drop_highly_correlated_features, scale_and_encode_final, ppccaa, find_task_type
)


df = pd.read_csv('train.csv')
target = 'SalePrice'
task_type = find_task_type(df, target)

print(f"Исходный размер: {df.shape[0]} строк, {df.shape[1]} колонок")
print(f"Всего пропусков до старта: {df.isnull().sum().sum()}")

Целевая переменная 'SalePrice' определена как РЕГРЕССИЯ.
Исходный размер: 1460 строк, 81 колонок
Всего пропусков до старта: 6965


## Шаг 1: Приведение типов и удаление мусора
Сначала конвертируем скрытые категории (числа, у которых меньше 10 уникальных значений) в текст. Затем удаляем технический столбец `Id`, автоматически распознаем даты (например, `YrSold`) и сбрасываем признаки, у которых одно значение занимает более 95% датасета и никак не влияет на таргет.

In [ ]:
df = convert_hidden_categories(df, target)

df = drop_columns_by_index(df, [1])

df = parse_dt(df)

# умный фильтр вариативности (частота > 95%, влияние на таргет < 10%)
df = drop_useless_columns_smartly(df, target, task_type, om=0.95, n=0.10)


--- Поиск скрытых категорий ---

Итого преобразовано столбцов в текст: 11
--- Поиск и обработка дат ---
Колонка 'YrSold' распознана как дата.
Удаление излишних данных (маленькая вариативность и отсутствие корреляции с целевой переменной)
Найдено и удалено бесполезных столбцов: 5


## Шаг 2: Обработка пропусков (почти 7000 NaN)
Вместо грубого `dropna()`, мы применяем умную стратегию:
1. Выделяем пропуски в категориальных признаках в отдельный класс `None` (часто отсутствие гаража — это важный сигнал, а не просто потерянные данные).
2. Оставшиеся числовые пропуски заливаем безопасной медианой, а текстовые — модой.

In [ ]:
# выделяем пропуски в категориальных фичах
df = fill_nan_with_new_class(df)

# медиана и мода
df = fill_nan_smart_impute(df)

print(f"Осталось пропусков: {df.isnull().sum().sum()}")

пропуски выделены
Все столбцы заполнены (мода/медиана/ffill).
Осталось пропусков: 0


## Шаг 3: Выбросы и отбор признаков (Feature Selection)
Чистим датасет от аномалий с помощью `IsolationForest`. Затем оцениваем полезность каждого признака: удаляем числовые колонки со слабой корреляцией и текстовые с низким показателем Mutual Information. В конце проверяем матрицу на мультиколлинеарность (порог 0.85).

In [ ]:
# двумерная очистка выбросов (отрезаем 3% самых жестких аномалий)
df = two_dimens(df, target, cont_pct=0.03)

# удаление признаков со слабой связью к таргету
df = zero_corr(df, target, task_type, corrind=0.1, mi_ind=0.03)

# защита от мультиколлинеарности 
df = drop_highly_correlated_features(df, target, threshold=0.85)

print(f"Размер после жесткой чистки: {df.shape[0]} строк, {df.shape[1]} колонок")

Изолирующий лес нашел и удалил 44 строк.
Удаление данных, которые не коррелируют
Удалено нерелевантных столбцов: 28

--- Удаление дублирующих признаков (Мультиколлинеарность) ---
Удалено 0 дублирующих признаков.
Размер после жесткой чистки: 1416 строк, 53 колонок


## Шаг 4: Кодирование, масштабирование и PCA
Финальный этап подготовки к ML: нормализуем распределение чисел (StandardScaler) и превращаем текст в бинарные флаги (One-Hot Encoding). Из-за OHE колонок стало слишком много, поэтому сжимаем датасет алгоритмом PCA, сохранив 90% полезной дисперсии.

In [ ]:
# OHE + StandardScaler (масштабируем все для pca)
df = scale_and_encode_final(df, target, scale_all=True)

# PCA (сжатие до 90% информации)
df = ppccaa(df, target, d=0.90)

print(f"ИТОГ: Готовая матрица на {df.shape[0]} строк и {df.shape[1]} колонок.")
display(df.head(3))


масштабирование и кодирование
Отмасштабировано 18 числовых колонок.
Текстовые признаки превращены в бинарные (One-Hot Encoding).

Готово! Итоговый размер таблицы: 1416 строк, 207 колонок.

--- Метод главных компонент (PCA) ---
Признаков до сжатия: 206
Осталось главных компонент после PCA: 45
ИТОГ: Готовая матрица на 1416 строк и 46 колонок.


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC37,PC38,PC39,PC40,PC41,PC42,PC43,PC44,PC45,SalePrice
0,2.465625,1.072153,-1.853266,-1.774105,0.434753,-0.229356,-0.923026,-0.910092,-0.425278,-0.461965,...,0.084291,-0.103487,0.059166,-0.081211,-0.061190,-0.343220,-0.292849,0.042593,-0.189380,208500
1,0.128197,-1.806745,0.943653,-1.080717,-0.934270,0.224636,1.459763,0.234097,-0.477708,-0.065090,...,-0.383845,0.213124,-0.094284,0.377982,-0.045630,-0.009794,0.087971,-0.664048,-0.183407,181500
2,2.650188,0.802299,-1.686411,-1.236347,0.020315,-0.487540,-0.621317,-1.163735,0.192740,-0.152702,...,-0.072009,0.196803,-0.207965,-0.028490,0.119682,-0.341243,-0.159307,-0.143709,-0.170292,223500
